# Демонстрационный ноутбук: модель v2.0 (Запрос 8)

Сквозной сценарий v2.0 реализован через монолитные процессы: каждая система — отдельный `multiprocessing.Process`.
Взаимодействие идёт через встроенный брокер **ВАБС** (`notebooks/sbd-model-demo-code/broker.py`) и роутинг по полю `receiver`.

Ниже встроены диаграммы последовательности сценария:

![Сквозной сценарий](../docs/integration_process/diagrams/10-e2e-agro-drone.png)


## Flow сценария (какие сообщения показываются)

![1) Заказчик → Агрегатор: receive_order](../docs/integration_process/diagrams/01-customer-aggregator-receive_order.png)

![2) Агрегатор → Эксплуатант: proposal_request](../docs/integration_process/diagrams/02-aggregator-operator-receive_order-proposal.png)

![3) Эксплуатант → Страховщик: request_insurance_quote](../docs/integration_process/diagrams/03-operator-insurer-request_insurance_quote.png)

![4) Планирование миссии (НУС)](../docs/integration_process/diagrams/06-operator-nus-mission_details.png)

![5) ОрВД БАС: validate_mission + разрешение на вылет](../docs/integration_process/diagrams/09-operator-atm-validate_mission.png)

![6) Сквозной end-to-end цикл](../docs/integration_process/diagrams/10-e2e-agro-drone.png)


## Документация по сущностям (монолиты)

В реализации ниже используются классы из `notebooks/sbd-model-demo-code/entities/`.

**CustomerEntity** — принимает `receive_order`, форвардит в `aggregator`.

**AggregatorEntity** — формирует `order_security_goals`, собирает `proposal_request` от исполнителей и выбирает исполнителя.

**OperatorEntity** — (Эксплуатант) выбирает БАС по данным от `droneport`, запрашивает quote у `insurer`, затем исполняет заказ: planning в НУС, согласование в ОрВД БАС, подготовка dispatch, передача миссии Агро-дрону.

**DronePortEntity** — готовность БАС/dispatch и выдача landing-кординат на `return_port`.

**NUSEntity** — формирует `mission_details` (включая `return_port` и `landing_coordinates`).

**ATMEntity** — согласует миссию и выдаёт разрешение на вылет после validate.

**AgroDroneEntity** — после разрешений и check Дронопорта запрашивает landing и возвращает `landing_coordinates`.


In [ ]:
import sys
import time
import multiprocessing as mp
from pathlib import Path
from typing import Any, Dict, List, Optional
import queue as pyqueue


def _find_repo_root(start: Path) -> Path:
    root = start
    for _ in range(8):
        if (root / "broker").exists() and (root / "systems").exists():
            return root
        if root.parent == root:
            break
        root = root.parent
    return start


repo_root = _find_repo_root(Path.cwd())
sbd_demo_code_dir = repo_root / "notebooks" / "sbd-model-demo-code"
sys.path.insert(0, str(sbd_demo_code_dir))

from actions import PLACE_ORDER  # noqa: E402
from broker import VABSBroker  # noqa: E402
from entities import (  # noqa: E402
    AggregatorEntity,
    ATMEntity,
    AgroDroneEntity,
    CustomerEntity,
    DronePortEntity,
    OperatorEntity,
    InsurerEntity,
    NUSEntity,
    RegulatorEntity,
    SITLEntity,
    DevelopersEntity,
)
from messages import (  # noqa: E402
    BROKER_STOP_ACTION,
    STOP_ACTION,
    make_request,
    new_correlation_id,
)


In [ ]:
def build_world() -> Dict[str, Any]:
    return {
        "aggregator_constants": {"agro": ["SG_ID_CONSTR_010"]},
        "operator_ids": ["operator_1", "operator_2"],
        "system_security_goals": {
            "operator_1": ["SG_ID_AUTH_001", "SG_ID_SAF_002", "SG_ID_CONSTR_010"],
            "operator_2": ["SG_ID_AUTH_001", "SG_ID_SAF_002"],
        },
        "operators": {
            "operator_1": {"system_security_goals": ["SG_ID_AUTH_001", "SG_ID_SAF_002", "SG_ID_CONSTR_010"]},
            "operator_2": {"system_security_goals": ["SG_ID_AUTH_001", "SG_ID_SAF_002"]},
        },
        "default_return_port": "droneport_B",
        "dronports": {
            "droneport_A": {
                "uas": [
                    {"uas_id": "uas_A_01", "model_id": "agro-model-1", "supported_task_types": ["agro"], "base_cost": 9000.0},
                    {"uas_id": "uas_A_02", "model_id": "agro-model-2", "supported_task_types": ["agro"], "base_cost": 10500.0},
                ]
            },
            "droneport_B": {
                "uas": [
                    {"uas_id": "uas_B_11", "model_id": "agro-model-3", "supported_task_types": ["agro"], "base_cost": 9500.0},
                ]
            },
        },
        "landing_sites": {
            "ORDER-DEMO-001": {
                "droneport_A": [55.75, 37.61],
                "droneport_B": [55.76, 37.62],
            }
        },
    }


In [3]:
def _wait_for_rpc_main(
    reply_queue: Any,
    *,
    correlation_id: str,
    expected_sender: Optional[str],
    timeout_s: float,
) -> Dict[str, Any]:
    deadline = time.time() + timeout_s
    while True:
        if time.time() > deadline:
            raise TimeoutError(
                f"Timeout waiting response corr={correlation_id} expected_sender={expected_sender}"
            )
        try:
            msg = reply_queue.get(timeout=0.2)
        except Exception:
            continue
        if not isinstance(msg, dict):
            continue
        if msg.get("correlation_id") != correlation_id:
            continue
        if expected_sender is not None and msg.get("sender") != expected_sender:
            continue
        return msg.get("payload", {})


## Запуск ВАБС и процессов

В этой среде `multiprocessing.Queue` создаёт `PermissionError`, поэтому для IPC используем `multiprocessing.Pipe` через thin-wrapper, совместимый по интерфейсу с `get/put/get_nowait`.


In [4]:
class PipeQueue:
    """Thin adapter: Connection -> Queue-like интерфейс."""

    def __init__(self, conn: Any):
        self._conn = conn

    def put(self, item: Any) -> None:
        self._conn.send(item)

    def get_nowait(self) -> Any:
        if self._conn.poll(0):
            return self._conn.recv()
        raise pyqueue.Empty()

    def get(self, timeout: Optional[float] = None) -> Any:
        if timeout is None:
            return self._conn.recv()
        if self._conn.poll(timeout):
            return self._conn.recv()
        raise pyqueue.Empty()


def _reply_queue_name(entity_id: str) -> str:
    return f"reply_{entity_id}"


In [ ]:
def _launch_processes(world: Dict[str, Any], log_path: str):
    ctx = mp.get_context("fork")

    broker_conn_read, broker_conn_write = ctx.Pipe()
    broker_in_queue_read = PipeQueue(broker_conn_read)
    broker_in_queue_write = PipeQueue(broker_conn_write)

    entities_inbox_ids: List[str] = [
        "customer",
        "aggregator",
        "operator_1",
        "operator_2",
        "insurer",
        "developers",
        "droneport_A",
        "droneport_B",
        "nus",
        "agro_drone",
        "sitl",
        "atm",
        "regulator",
    ]

    inbox_queues: Dict[str, Any] = {}
    reply_queues: Dict[str, Any] = {}
    broker_inbox_send: Dict[str, Any] = {}
    broker_reply_send: Dict[str, Any] = {}

    for eid in entities_inbox_ids:
        conn_in_read, conn_in_write = ctx.Pipe()
        inbox_queues[eid] = PipeQueue(conn_in_read)
        broker_inbox_send[eid] = PipeQueue(conn_in_write)

        conn_reply_read, conn_reply_write = ctx.Pipe()
        reply_queues[eid] = PipeQueue(conn_reply_read)
        broker_reply_send[eid] = PipeQueue(conn_reply_write)

    orchestrator_reply_name = "reply_orchestrator"
    conn_orch_read, conn_orch_write = ctx.Pipe()
    orchestrator_reply_queue = PipeQueue(conn_orch_read)
    orchestrator_reply_send = PipeQueue(conn_orch_write)

    broker = VABSBroker(
        broker_in_queue=broker_in_queue_read,
        log_path=log_path,
        poll_sleep_s=0.01,
    )

    for eid in entities_inbox_ids:
        broker.register_queue(eid, broker_inbox_send[eid])
        broker.register_queue(_reply_queue_name(eid), broker_reply_send[eid])
    broker.register_queue(orchestrator_reply_name, orchestrator_reply_send)
    broker.start()

    procs: List[mp.Process] = []
    procs.append(
        CustomerEntity(entity_id="customer", inbox_queue=inbox_queues["customer"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["customer"], reply_queue_name=_reply_queue_name("customer"), world=world)
    )
    procs.append(
        AggregatorEntity(entity_id="aggregator", inbox_queue=inbox_queues["aggregator"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["aggregator"], reply_queue_name=_reply_queue_name("aggregator"), world=world)
    )

    for ex_id in ["operator_1", "operator_2"]:
        procs.append(
            OperatorEntity(entity_id=ex_id, inbox_queue=inbox_queues[ex_id], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues[ex_id], reply_queue_name=_reply_queue_name(ex_id), world=world)
        )

    procs.append(
        InsurerEntity(entity_id="insurer", inbox_queue=inbox_queues["insurer"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["insurer"], reply_queue_name=_reply_queue_name("insurer"), world=world)
    )
    procs.append(
        DevelopersEntity(entity_id="developers", inbox_queue=inbox_queues["developers"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["developers"], reply_queue_name=_reply_queue_name("developers"), world=world)
    )

    for dp_id in ["droneport_A", "droneport_B"]:
        procs.append(
            DronePortEntity(entity_id=dp_id, inbox_queue=inbox_queues[dp_id], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues[dp_id], reply_queue_name=_reply_queue_name(dp_id), world=world)
        )

    procs.append(
        NUSEntity(entity_id="nus", inbox_queue=inbox_queues["nus"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["nus"], reply_queue_name=_reply_queue_name("nus"), world=world)
    )
    procs.append(
        AgroDroneEntity(entity_id="agro_drone", inbox_queue=inbox_queues["agro_drone"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["agro_drone"], reply_queue_name=_reply_queue_name("agro_drone"), world=world)
    )
    procs.append(
        SITLEntity(entity_id="sitl", inbox_queue=inbox_queues["sitl"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["sitl"], reply_queue_name=_reply_queue_name("sitl"), world=world)
    )
    procs.append(
        ATMEntity(entity_id="atm", inbox_queue=inbox_queues["atm"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["atm"], reply_queue_name=_reply_queue_name("atm"), world=world)
    )
    procs.append(
        RegulatorEntity(entity_id="regulator", inbox_queue=inbox_queues["regulator"], broker_in_queue=broker_in_queue_write, reply_queue=reply_queues["regulator"], reply_queue_name=_reply_queue_name("regulator"), world=world)
    )

    for p in procs:
        p.start()

    return (broker_in_queue_write, broker, orchestrator_reply_queue, orchestrator_reply_name, entities_inbox_ids, procs)


def _shutdown(broker_in_queue_write: Any, broker: Any, entities_inbox_ids: List[str], procs: List[mp.Process]) -> None:
    for eid in entities_inbox_ids:
        broker_in_queue_write.put({"sender": "orchestrator", "receiver": eid, "action": STOP_ACTION, "payload": {}, "correlation_id": "stop", "message_type": "request"})
    broker_in_queue_write.put({"sender": "orchestrator", "receiver": "__broker__", "action": BROKER_STOP_ACTION, "payload": {}, "correlation_id": "stop", "message_type": "request"})
    for p in procs:
        p.join(timeout=5.0)
    broker.join(timeout=5.0)


## Выполнение сценария (один заказ)

![receive_order](../docs/integration_process/diagrams/01-customer-aggregator-receive_order.png)
![proposal_request](../docs/integration_process/diagrams/02-aggregator-operator-receive_order-proposal.png)
![insurance quote](../docs/integration_process/diagrams/03-operator-insurer-request_insurance_quote.png)
![validate mission](../docs/integration_process/diagrams/09-operator-atm-validate_mission.png)

Проверка корректности: `landing_coordinates` должны совпасть с `world["landing_sites"][order["id"]][order["return_port"]]`.


In [6]:
def _run_single_order(
    broker_in_queue_write: Any,
    orchestrator_reply_queue: Any,
    orchestrator_reply_name: str,
    *,
    order: Dict[str, Any],
    scenario_security_goals: List[str],
    max_price: float,
    timeout_s: float = 180.0,
) -> Dict[str, Any]:
    correlation_id = new_correlation_id()
    initial_msg = make_request(
        sender="orchestrator",
        receiver="customer",
        action=PLACE_ORDER,
        payload={
            "order": order,
            "scenario_security_goals": scenario_security_goals,
            "max_price": max_price,
        },
        correlation_id=correlation_id,
        reply_to=orchestrator_reply_name,
    ).to_dict()
    broker_in_queue_write.put(initial_msg)
    return _wait_for_rpc_main(
        orchestrator_reply_queue,
        correlation_id=correlation_id,
        expected_sender="customer",
        timeout_s=timeout_s,
    )


In [ ]:
def run_demo() -> Dict[str, Any]:
    world = build_world()
    log_path = str(repo_root / "notebooks" / "simulation.log")

    (
        broker_in_queue_write,
        broker,
        orchestrator_reply_queue,
        orchestrator_reply_name,
        entities_inbox_ids,
        procs,
    ) = _launch_processes(world, log_path=log_path)

    try:
        order = {
            "id": "ORDER-DEMO-001",
            "scenario_type": "agro",
            "destination": {"lat": 55.75, "lon": 37.61},
            "return_port": "droneport_B",
            "coverage": {"min_payload": 3.5, "min_range": 1.0, "min_battery": 0.8},
        }
        scenario_security_goals = ["SG_ID_AUTH_001", "SG_ID_SAF_002"]
        max_price = 999999.0
        final_payload = _run_single_order(
            broker_in_queue_write,
            orchestrator_reply_queue,
            orchestrator_reply_name,
            order=order,
            scenario_security_goals=scenario_security_goals,
            max_price=max_price,
        )

        assert final_payload.get("status") == "ok", final_payload
        assert "selected_operator" in final_payload
        chosen = final_payload.get("chosen_proposal") or {}
        order_goals = final_payload.get("order_security_goals") or []
        applied_goals = chosen.get("applied_security_goals") or []
        assert set(order_goals).issubset(set(applied_goals)), (order_goals, applied_goals)

        completed = final_payload.get("order_execution_completed")
        assert completed is not None
        coords = completed.get("landing_coordinates")
        assert coords is not None and isinstance(coords, list) and len(coords) == 2, coords
        expected_coords = world["landing_sites"][order["id"]][order["return_port"]]
        assert coords == expected_coords, (coords, expected_coords)
        return final_payload
    finally:
        _shutdown(broker_in_queue_write, broker, entities_inbox_ids, procs)


In [8]:
try:
    demo_result = run_demo()
    print("DEMO FINISHED")
    demo_result
except Exception as e:  # noqa: BLE001
    print("DEMO FAILED:", type(e).__name__, str(e))
    raise


[broker] start pid=3250470 log=/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/notebooks/simulation.log
[broker] |-| orchestrator->customer |-| action=place_order |-| request |-| corr=6c772072589d40618e25ec505007dd90 |---| payload={'order': {'id': 'ORDER-DEMO-001', 'scenario_type': 'agro', 'destination': {'lat': 55.75, 'lon': 37.61}, 'return_port': 'droneport_B', 'coverage': {'min_payload': 3.5, 'min_range': 1.0, 'min_battery': 0.8}}, 'scenario_security_goals': ['SG_ID_AUTH_001', 'SG_ID_SAF_002'], 'max_price': 999999.0}
[broker] |-| customer->aggregator |-| action=receive_order |-| request |-| corr=f221884dab1645a1aabcb0824c211c06 |---| payload={'order': {'id': 'ORDER-DEMO-001', 'scenario_type': 'agro', 'destination': {'lat': 55.75, 'lon': 37.61}, 'return_port': 'droneport_B', 'coverage': {'min_payload': 3.5, 'min_range': 1.0, 'min_battery': 0.8}}, 'scenario_security_goals': ['SG_ID_AUTH_001', 'SG_ID_SAF_002'], 'max_price': 999999.0}
[broker] |-| aggregator->executor_1